## Fase #2: Comprensión de los Datos / Data Understanding
La fase de comprensión de datos comienza con una recopilación inicial de datos y continúa con actividades para familiarizarse con los datos, identificar problemas de calidad de los datos, descubrir los primeros conocimientos sobre los datos o detectar subconjuntos interesantes para formular hipótesis sobre información oculta.
* Establecé una metodología de adquisición de los datos para el negocio.
* Contemplá en la adquisición de datos una alternativa donde debas construir el setde datos.
* Identificá los distintos problemas o desafíos con los datos (datos faltantes, ruidos,etc) que puedan afectar a su calidad.
* Enumerá las distintas fases, tareas generales, tareas específicas e instancias del proceso que incluirás como estrategia para aplicar la metodología

### Fuentes de los datos

### Forma de obtención

### Condiciones de uso y licencias

Adquisición, construcción y calidad ---> VER DONDE PONER ESTO XD

**Fuente principal.** Se descarga manualmente el CSV de actualización diaria del Adjunto IV y se conserva sin editar. Cada ejecución registra nombre, tamaño, SHA-256 y fecha máxima de `fecha_data`; así se puede identificar exactamente la versión utilizada.

**Alternativa construida.** Como extensión, se puede unir el Adjunto IV con la producción no convencional mediante `idpozo` y agregar la producción de los primeros seis meses posteriores a `fecha_fin_fractura`. Ese conjunto permitiría estudiar producción inicial según características de completación. Requiere resolver pozos con múltiples intervenciones, meses faltantes y ventanas incompletas, por lo que no forma parte del modelo principal.

**Desafíos de calidad.** Se controlarán duplicados de `id_base_fractura_adjiv`, valores no numéricos, ceros, fechas inválidas o invertidas, categorías poco frecuentes, pozos repetidos y valores extremos. Los filtros se aplican de manera secuencial para que el resumen de exclusiones sea reproducible. No se imputará la variable objetivo.

In [7]:
import hashlib
from pathlib import Path
from urllib.request import urlretrieve
import requests
import urllib3

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display  # ty: ignore[unresolved-import]


### Descarga de archivo

In [9]:
#@title Descargar datos de energía
# Silenciar advertencias de certificados no verificados
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

url = "https://datos.energia.gob.ar/dataset/71fa2e84-0316-4a1b-af68-7f35e41f58d7/resource/2280ad92-6ed3-403e-a095-50139863ab0d/download/datos-de-fractura-de-pozos-de-hidrocarburos-adjunto-iv-actualizacin-diaria.csv"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# Realizar la descarga en bloques (stream) para evitar sobrecargar memoria
with requests.get(url, headers=headers, verify=False, stream=True, timeout=60) as r:
    r.raise_for_status()
    with open("datos_energia.csv", "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

print("Descarga completada con éxito.")


Descarga completada con éxito.


### Carga de datos

In [12]:
# @title Cargar datos
# Cargar el archivo CSV descargado en un DataFrame de pandas
df_datos_energia = pd.read_csv("datos_energia.csv")

# Visualizar las primeras filas del DataFrame
df_datos_energia.head()

,id_base_fractura_adjiv,idpozo,sigla,cuenca,areapermisoconcesion,yacimiento,formacion_productiva,tipo_reservorio,subtipo_reservorio,longitud_rama_horizontal_m,...,fecha_data,anio_if,mes_if,anio_ff,mes_ff,anio_carga,mes_carga,empresa_informante,mes,anio
0,30,159910,APS.Nq.ADC.xp-1033,NEUQUINA,AGUA DEL CAJON,AGUA DEL CAJON,los molles,NO CONVENCIONAL,SHALE,0.0,...,2019-06-14 17:13:03.68279,2019,4,2019,4,2019,6,CAPEX S.A.,4,2019
1,31,159910,APS.Nq.ADC.xp-1033,NEUQUINA,AGUA DEL CAJON,AGUA DEL CAJON,los molles,NO CONVENCIONAL,SHALE,0.0,...,2019-06-14 17:14:19.179874,2018,11,2018,11,2019,6,CAPEX S.A.,11,2018
2,37,159219,YPF.Nq.AdlA-1001(h),NEUQUINA,AGUADA DE LA ARENA,AGUADA DE LA ARENA,vaca muerta,NO CONVENCIONAL,SHALE,1437.3,...,2019-06-27 13:46:21.14935,2017,11,2017,12,2019,6,YPF S.A.,11,2017
3,38,159220,YPF.Nq.AdlA-1002(h),NEUQUINA,AGUADA DE LA ARENA,AGUADA DE LA ARENA,vaca muerta,NO CONVENCIONAL,SHALE,1518.3,...,2019-06-27 13:46:21.14935,2017,11,2017,12,2019,6,YPF S.A.,11,2017
4,39,159221,YPF.Nq.AdlA-1003(h),NEUQUINA,AGUADA DE LA ARENA,AGUADA DE LA ARENA,vaca muerta,NO CONVENCIONAL,SHALE,1482.3,...,2019-06-27 13:46:21.14935,2017,11,2017,12,2019,6,YPF S.A.,11,2017


### Exploración inicial

In [15]:
# Determinar el número de filas y columnas del dataset
num_filas, num_columnas = df_datos_energia.shape

# Mostrar el número de filas y columnas
print(f"El Dataset tiene {num_filas} filas y {num_columnas} columnas.")

El Dataset tiene 4983 filas y 30 columnas.


In [ ]:

# Definición del diccionario con el mapeo de variables --FIIJARSE BIEN ESTO XD---
mapeo_variables = [
    {
        "Variable Original": "id_base_fractura_adjiv",
        "Tipo Actual (Pandas)": str(df_datos_energia["id_base_fractura_adjiv"].dtype),
        "Nombre Propuesto": "id_fractura",
        "Tipo Correspondiente": "Categórico / Identificador",
        "Dtype Sugerido": "string",
        "Valores Nulos": int(df_datos_energia["id_base_fractura_adjiv"].isnull().sum()),
        "Observaciones / Acción": "Clave primaria del registro. No operar matemáticamente."
    },
    {
        "Variable Original": "idpozo",
        "Tipo Actual (Pandas)": str(df_datos_energia["idpozo"].dtype),
        "Nombre Propuesto": "id_pozo",
        "Tipo Correspondiente": "Categórico / Identificador",
        "Dtype Sugerido": "string",
        "Valores Nulos": int(df_datos_energia["idpozo"].isnull().sum()),
        "Observaciones / Acción": "Identificador único de pozo (foreign key)."
    },
    {
        "Variable Original": "sigla",
        "Tipo Actual (Pandas)": str(df_datos_energia["sigla"].dtype),
        "Nombre Propuesto": "pozo_sigla",
        "Tipo Correspondiente": "Categórico nominal",
        "Dtype Sugerido": "category",
        "Valores Nulos": int(df_datos_energia["sigla"].isnull().sum()),
        "Observaciones / Acción": "Código o nombre identificador del pozo."
    },
    {
        "Variable Original": "cuenca",
        "Tipo Actual (Pandas)": str(df_datos_energia["cuenca"].dtype),
        "Nombre Propuesto": "cuenca",
        "Tipo Correspondiente": "Categórico nominal",
        "Dtype Sugerido": "category",
        "Valores Nulos": int(df_datos_energia["cuenca"].isnull().sum()),
        "Observaciones / Acción": "Cuenca geológica (baja cardinalidad)."
    },
    {
        "Variable Original": "areapermisoconcesion",
        "Tipo Actual (Pandas)": str(df_datos_energia["areapermisoconcesion"].dtype),
        "Nombre Propuesto": "area_concesion",
        "Tipo Correspondiente": "Categórico nominal",
        "Dtype Sugerido": "category",
        "Valores Nulos": int(df_datos_energia["areapermisoconcesion"].isnull().sum()),
        "Observaciones / Acción": "Bloque o área hidrocarburífera de concesión."
    },
    {
        "Variable Original": "yacimiento",
        "Tipo Actual (Pandas)": str(df_datos_energia["yacimiento"].dtype),
        "Nombre Propuesto": "yacimiento",
        "Tipo Correspondiente": "Categórico nominal",
        "Dtype Sugerido": "category",
        "Valores Nulos": int(df_datos_energia["yacimiento"].isnull().sum()),
        "Observaciones / Acción": "Yacimiento productivo."
    },
    {
        "Variable Original": "formacion_productiva",
        "Tipo Actual (Pandas)": str(df_datos_energia["formacion_productiva"].dtype),
        "Nombre Propuesto": "formacion_productiva",
        "Tipo Correspondiente": "Categórico nominal",
        "Dtype Sugerido": "category",
        "Valores Nulos": int(df_datos_energia["formacion_productiva"].isnull().sum()),
        "Observaciones / Acción": "Formación objetivo (ej. Vaca Muerta)."
    },
    {
        "Variable Original": "tipo_reservorio",
        "Tipo Actual (Pandas)": str(df_datos_energia["tipo_reservorio"].dtype),
        "Nombre Propuesto": "tipo_reservorio",
        "Tipo Correspondiente": "Categórico nominal",
        "Dtype Sugerido": "category",
        "Valores Nulos": int(df_datos_energia["tipo_reservorio"].isnull().sum()),
        "Observaciones / Acción": "Convencional / No Convencional. Evaluar imputación de nulos."
    },
    {
        "Variable Original": "subtipo_reservorio",
        "Tipo Actual (Pandas)": str(df_datos_energia["subtipo_reservorio"].dtype),
        "Nombre Propuesto": "subtipo_reservorio",
        "Tipo Correspondiente": "Categórico nominal",
        "Dtype Sugerido": "category",
        "Valores Nulos": int(df_datos_energia["subtipo_reservorio"].isnull().sum()),
        "Observaciones / Acción": "Shale / Tight. 973 nulos a tratar en Fase 3."
    },
    {
        "Variable Original": "longitud_rama_horizontal_m",
        "Tipo Actual (Pandas)": str(df_datos_energia["longitud_rama_horizontal_m"].dtype),
        "Nombre Propuesto": "longitud_rama_horizontal_m",
        "Tipo Correspondiente": "Numérico continuo",
        "Dtype Sugerido": "float64",
        "Valores Nulos": int(df_datos_energia["longitud_rama_horizontal_m"].isnull().sum()),
        "Observaciones / Acción": "Longitud perforada en metros."
    },
    {
        "Variable Original": "cantidad_fracturas",
        "Tipo Actual (Pandas)": str(df_datos_energia["cantidad_fracturas"].dtype),
        "Nombre Propuesto": "etapas_fractura",
        "Tipo Correspondiente": "Numérico discreto",
        "Dtype Sugerido": "int64",
        "Valores Nulos": int(df_datos_energia["cantidad_fracturas"].isnull().sum()),
        "Observaciones / Acción": "Cantidad de etapas estimuladas (stages)."
    },
    {
        "Variable Original": "tipo_terminacion",
        "Tipo Actual (Pandas)": str(df_datos_energia["tipo_terminacion"].dtype),
        "Nombre Propuesto": "tipo_terminacion",
        "Tipo Correspondiente": "Categórico nominal",
        "Dtype Sugerido": "category",
        "Valores Nulos": int(df_datos_energia["tipo_terminacion"].isnull().sum()),
        "Observaciones / Acción": "Método técnico de terminación (Plug & Perf, etc.)."
    },
    {
        "Variable Original": "arena_bombeada_nacional_tn",
        "Tipo Actual (Pandas)": str(df_datos_energia["arena_bombeada_nacional_tn"].dtype),
        "Nombre Propuesto": "arena_nacional_tn",
        "Tipo Correspondiente": "Numérico continuo",
        "Dtype Sugerido": "float64",
        "Valores Nulos": int(df_datos_energia["arena_bombeada_nacional_tn"].isnull().sum()),
        "Observaciones / Acción": "Agente de sostén local inyectado (toneladas)."
    },
    {
        "Variable Original": "arena_bombeada_importada_tn",
        "Tipo Actual (Pandas)": str(df_datos_energia["arena_bombeada_importada_tn"].dtype),
        "Nombre Propuesto": "arena_importada_tn",
        "Tipo Correspondiente": "Numérico continuo",
        "Dtype Sugerido": "float64",
        "Valores Nulos": int(df_datos_energia["arena_bombeada_importada_tn"].isnull().sum()),
        "Observaciones / Acción": "Agente de sostén importado inyectado (toneladas)."
    },
    {
        "Variable Original": "agua_inyectada_m3",
        "Tipo Actual (Pandas)": str(df_datos_energia["agua_inyectada_m3"].dtype),
        "Nombre Propuesto": "agua_inyectada_m3",
        "Tipo Correspondiente": "Numérico continuo",
        "Dtype Sugerido": "float64",
        "Valores Nulos": int(df_datos_energia["agua_inyectada_m3"].isnull().sum()),
        "Observaciones / Acción": "Volumen de fluido base agua en metros cúbicos."
    },
    {
        "Variable Original": "co2_inyectado_m3",
        "Tipo Actual (Pandas)": str(df_datos_energia["co2_inyectado_m3"].dtype),
        "Nombre Propuesto": "co2_inyectado_m3",
        "Tipo Correspondiente": "Numérico continuo",
        "Dtype Sugerido": "float64",
        "Valores Nulos": int(df_datos_energia["co2_inyectado_m3"].isnull().sum()),
        "Observaciones / Acción": "Revisar si tiene varianza nula (solo ceros)."
    },
    {
        "Variable Original": "presion_maxima_psi",
        "Tipo Actual (Pandas)": str(df_datos_energia["presion_maxima_psi"].dtype),
        "Nombre Propuesto": "presion_maxima_psi",
        "Tipo Correspondiente": "Numérico continuo",
        "Dtype Sugerido": "float64",
        "Valores Nulos": int(df_datos_energia["presion_maxima_psi"].isnull().sum()),
        "Observaciones / Acción": "Presión pico de tratamiento en psi."
    },
    {
        "Variable Original": "potencia_equipos_fractura_hp",
        "Tipo Actual (Pandas)": str(df_datos_energia["potencia_equipos_fractura_hp"].dtype),
        "Nombre Propuesto": "potencia_equipos_hp",
        "Tipo Correspondiente": "Numérico continuo",
        "Dtype Sugerido": "float64",
        "Valores Nulos": int(df_datos_energia["potencia_equipos_fractura_hp"].isnull().sum()),
        "Observaciones / Acción": "Potencia instalada en caballos de fuerza (51 nulos)."
    },
    {
        "Variable Original": "fecha_inicio_fractura",
        "Tipo Actual (Pandas)": str(df_datos_energia["fecha_inicio_fractura"].dtype),
        "Nombre Propuesto": "fecha_inicio_fractura",
        "Tipo Correspondiente": "Temporal",
        "Dtype Sugerido": "datetime64[ns]",
        "Valores Nulos": int(df_datos_energia["fecha_inicio_fractura"].isnull().sum()),
        "Observaciones / Acción": "Convertir con pd.to_datetime."
    },
    {
        "Variable Original": "fecha_fin_fractura",
        "Tipo Actual (Pandas)": str(df_datos_energia["fecha_fin_fractura"].dtype),
        "Nombre Propuesto": "fecha_fin_fractura",
        "Tipo Correspondiente": "Temporal",
        "Dtype Sugerido": "datetime64[ns]",
        "Valores Nulos": int(df_datos_energia["fecha_fin_fractura"].isnull().sum()),
        "Observaciones / Acción": "Convertir con pd.to_datetime para medir duración."
    },
    {
        "Variable Original": "fecha_data",
        "Tipo Actual (Pandas)": str(df_datos_energia["fecha_data"].dtype),
        "Nombre Propuesto": "fecha_registro",
        "Tipo Correspondiente": "Temporal",
        "Dtype Sugerido": "datetime64[ns]",
        "Valores Nulos": int(df_datos_energia["fecha_data"].isnull().sum()),
        "Observaciones / Acción": "Fecha de reporte al sistema."
    },
    {
        "Variable Original": "anio_if",
        "Tipo Actual (Pandas)": str(df_datos_energia["anio_if"].dtype),
        "Nombre Propuesto": "anio_inicio_fractura",
        "Tipo Correspondiente": "Numérico / Ordinal",
        "Dtype Sugerido": "int32",
        "Valores Nulos": int(df_datos_energia["anio_if"].isnull().sum()),
        "Observaciones / Acción": "Redundante si se procesa fecha_inicio_fractura."
    },
    {
        "Variable Original": "mes_if",
        "Tipo Actual (Pandas)": str(df_datos_energia["mes_if"].dtype),
        "Nombre Propuesto": "mes_inicio_fractura",
        "Tipo Correspondiente": "Categórico / Ordinal",
        "Dtype Sugerido": "int32",
        "Valores Nulos": int(df_datos_energia["mes_if"].isnull().sum()),
        "Observaciones / Acción": "Redundante con la columna temporal."
    },
    {
        "Variable Original": "anio_ff",
        "Tipo Actual (Pandas)": str(df_datos_energia["anio_ff"].dtype),
        "Nombre Propuesto": "anio_fin_fractura",
        "Tipo Correspondiente": "Numérico / Ordinal",
        "Dtype Sugerido": "int32",
        "Valores Nulos": int(df_datos_energia["anio_ff"].isnull().sum()),
        "Observaciones / Acción": "Redundante si se procesa fecha_fin_fractura."
    },
    {
        "Variable Original": "mes_ff",
        "Tipo Actual (Pandas)": str(df_datos_energia["mes_ff"].dtype),
        "Nombre Propuesto": "mes_fin_fractura",
        "Tipo Correspondiente": "Categórico / Ordinal",
        "Dtype Sugerido": "int32",
        "Valores Nulos": int(df_datos_energia["mes_ff"].isnull().sum()),
        "Observaciones / Acción": "Redundante con la columna temporal."
    },
    {
        "Variable Original": "anio_carga",
        "Tipo Actual (Pandas)": str(df_datos_energia["anio_carga"].dtype),
        "Nombre Propuesto": "anio_carga_sistema",
        "Tipo Correspondiente": "Numérico / Ordinal",
        "Dtype Sugerido": "int32",
        "Valores Nulos": int(df_datos_energia["anio_carga"].isnull().sum()),
        "Observaciones / Acción": "Metadato administrativo de carga."
    },
    {
        "Variable Original": "mes_carga",
        "Tipo Actual (Pandas)": str(df_datos_energia["mes_carga"].dtype),
        "Nombre Propuesto": "mes_carga_sistema",
        "Tipo Correspondiente": "Categórico / Ordinal",
        "Dtype Sugerido": "int32",
        "Valores Nulos": int(df_datos_energia["mes_carga"].isnull().sum()),
        "Observaciones / Acción": "Metadato administrativo de carga."
    },
    {
        "Variable Original": "empresa_informante",
        "Tipo Actual (Pandas)": str(df_datos_energia["empresa_informante"].dtype),
        "Nombre Propuesto": "operadora",
        "Tipo Correspondiente": "Categórico nominal",
        "Dtype Sugerido": "category",
        "Valores Nulos": int(df_datos_energia["empresa_informante"].isnull().sum()),
        "Observaciones / Acción": "Compañía operadora del pozo."
    },
    {
        "Variable Original": "mes",
        "Tipo Actual (Pandas)": str(df_datos_energia["mes"].dtype),
        "Nombre Propuesto": "mes_periodo",
        "Tipo Correspondiente": "Categórico / Ordinal",
        "Dtype Sugerido": "int32",
        "Valores Nulos": int(df_datos_energia["mes"].isnull().sum()),
        "Observaciones / Acción": "Posible columna redundante a contrastar."
    },
    {
        "Variable Original": "anio",
        "Tipo Actual (Pandas)": str(df_datos_energia["anio"].dtype),
        "Nombre Propuesto": "anio_periodo",
        "Tipo Correspondiente": "Numérico / Ordinal",
        "Dtype Sugerido": "int32",
        "Valores Nulos": int(df_datos_energia["anio"].isnull().sum()),
        "Observaciones / Acción": "Posible columna redundante a contrastar."
    }
]

# Crear el DataFrame de metadatos
df_diccionario = pd.DataFrame(mapeo_variables)

# Mostrar las 30 filas completas en el notebook
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

df_diccionario

,Variable Original,Tipo Actual (Pandas),Nombre Propuesto,Tipo Correspondiente,Dtype Sugerido,Valores Nulos,Observaciones / Acción
0,id_base_fractura_adjiv,int64,id_fractura,Categórico / Identificador,string,0,Clave primaria del registro. No operar matemáticamente.
1,idpozo,int64,id_pozo,Categórico / Identificador,string,0,Identificador único de pozo (foreign key).
2,sigla,object,pozo_sigla,Categórico nominal,category,0,Código o nombre identificador del pozo.
3,cuenca,object,cuenca,Categórico nominal,category,0,Cuenca geológica (baja cardinalidad).
4,areapermisoconcesion,object,area_concesion,Categórico nominal,category,0,Bloque o área hidrocarburífera de concesión.
5,yacimiento,object,yacimiento,Categórico nominal,category,0,Yacimiento productivo.
6,formacion_productiva,object,formacion_productiva,Categórico nominal,category,0,Formación objetivo (ej. Vaca Muerta).
7,tipo_reservorio,object,tipo_reservorio,Categórico nominal,category,35,Convencional / No Convencional. Evaluar imputación de nulos.
8,subtipo_reservorio,object,subtipo_reservorio,Categórico nominal,category,973,Shale / Tight. 973 nulos a tratar en Fase 3.
9,longitud_rama_horizontal_m,float64,longitud_rama_horizontal_m,Numérico continuo,float64,0,Longitud perforada en metros.


In [ ]:
# Definimos las descripciones para armar la tabla
descripciones = {
    'Area Code': 'Código identificador del área geográfica',
    'Area Code (M49)': 'Código geográfico M49',
    'Area': 'País o región',
    'Item Code': 'Código de la actividad / producto',
    'Item': 'Actividad o categoría asociada a la emisión',
    'Element Code': 'Código del elemento medido',
    'Element': 'Tipo de emisión o gas registrado',
    'Year Code': 'Código correspondiente al año',
    'Year': 'Año de la observación',
    'Source Code': 'Código de la fuente',
    'Source': 'Fuente del dato',
    'Unit': 'Unidad de medida',
    'Value': 'Valor de la emisión',
    'Flag': 'Indicador sobre la naturaleza / calidad del valor',
    'Note': 'Observación o nota'
}

# Definimos los tipos de datos para armar la tabla
tipos_datos = {
    'Area Code': 'Categórica nominal',
    'Area Code (M49)': 'Categórica nominal',
    'Area': 'Categórica nominal',
    'Item Code': 'Categórica nominal',
    'Item': 'Categórica nominal',
    'Element Code': 'Categórica nominal',
    'Element': 'Categórica nominal',
    'Year Code': 'Categórica nominal',
    'Year': 'Cuantitativa discreta - Temporal',
    'Source Code': 'Categórica nominal',
    'Source': 'Categórica nominal',
    'Unit': 'Categórica nominal',
    'Value': 'Cuantitativa continua',
    'Flag': 'Categórica nominal',
    'Note': 'Categórica nominal'
}

In [ ]:
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)


sha256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()

columnas_numericas = [
	"longitud_rama_horizontal_m",
	"cantidad_fracturas",
	"agua_inyectada_m3",
	"arena_bombeada_nacional_tn",
	"arena_bombeada_importada_tn",
	"co2_inyectado_m3",
	"presion_maxima_psi",
	"potencia_equipos_fractura_hp",
]
columnas_fecha = ["fecha_inicio_fractura", "fecha_fin_fractura", "fecha_data"]

df = df_raw.copy()
for col in columnas_numericas:
	df[col] = pd.to_numeric(df[col], errors="coerce")
for col in columnas_fecha:
	df[col] = pd.to_datetime(df[col], errors="coerce")

pasos = []


def aplicar_filtro(nombre, condicion):
	global df
	antes = len(df)
	df = df.loc[condicion].copy()
	pasos.append({"paso": nombre, "conservados": len(df), "excluidos_en_paso": antes - len(df)})


pasos.append({"paso": "Archivo original", "conservados": len(df), "excluidos_en_paso": 0})
aplicar_filtro("Cuenca Neuquina", df["cuenca"].eq("NEUQUINA"))
aplicar_filtro("Reservorio no convencional", df["tipo_reservorio"].eq("NO CONVENCIONAL"))
aplicar_filtro("Rama horizontal > 0", df["longitud_rama_horizontal_m"].gt(0))
aplicar_filtro("Cantidad de fracturas > 0", df["cantidad_fracturas"].gt(0))
aplicar_filtro("Agua inyectada > 0", df["agua_inyectada_m3"].gt(0))
aplicar_filtro("Fechas de operación válidas", df[["fecha_inicio_fractura", "fecha_fin_fractura"]].notna().all(axis=1))
aplicar_filtro("Fin no anterior al inicio", df["fecha_fin_fractura"].ge(df["fecha_inicio_fractura"]))

duplicados_id = int(df["id_base_fractura_adjiv"].duplicated().sum())
if duplicados_id:
	raise ValueError(f"Hay {duplicados_id} identificadores de fractura duplicados; deben revisarse.")

perfil_fuente = pd.DataFrame(
	{
		"indicador": ["archivo", "tamaño_bytes", "sha256", "filas_originales", "columnas", "fecha_data_máxima", "pozos_en_muestra"],
		"valor": [
			DATA_PATH.name,
			DATA_PATH.stat().st_size,
			sha256,
			len(df_raw),
			df_raw.shape[1],
			df_raw["fecha_data"].pipe(pd.to_datetime, errors="coerce").max(),
			df["idpozo"].nunique(),
		],
	}
)
display(perfil_fuente)
display(pd.DataFrame(pasos))
print(f"Conjunto analítico: {len(df):,} registros de {df['idpozo'].nunique():,} pozos.")

,indicador,valor
0,archivo,datos-de-fractura-de-pozos-de-hidrocarburos-ad...
1,tamaño_bytes,1255065
2,sha256,72b5204463dff0f8a4f951f893344abb8992a540caa8fc...
3,filas_originales,4983
4,columnas,30
5,fecha_data_máxima,2026-10-07 08:43:27.073465
6,pozos_en_muestra,2666


,paso,conservados,excluidos_en_paso
0,Archivo original,4983,0
1,Cuenca Neuquina,4180,803
2,Reservorio no convencional,3805,375
3,Rama horizontal > 0,2734,1071
4,Cantidad de fracturas > 0,2734,0
5,Agua inyectada > 0,2732,2
6,Fechas de operación válidas,2732,0
7,Fin no anterior al inicio,2700,32


Conjunto analítico: 2,700 registros de 2,666 pozos.


### Interpretación de la adquisición y la calidad

El archivo analizado contiene **4.910 registros y 30 variables**. La huella SHA-256 permite reconocer esta versión exacta del CSV, cuya fecha máxima de carga es el **29/09/2026**. Después de aplicar todos los criterios quedan **2.626 registros de 2.592 pozos**. La diferencia entre registros y pozos confirma que algunos pozos tienen más de una intervención; por eso `idpozo` se conserva como control y no como variable predictora.

Los filtros son secuenciales: `excluidos_en_paso` se interpreta respecto del paso anterior, no del archivo completo. Primero se retiraron 803 registros de otras cuencas y luego 376 correspondientes a reservorios convencionales dentro de la cuenca Neuquina. La mayor reducción posterior proviene de **1.071 registros sin una rama horizontal positiva**, coherente con el alcance limitado a pozos horizontales. También se excluyeron dos registros sin agua positiva y 32 con fecha de finalización anterior a la fecha de inicio.

El conjunto final representa aproximadamente el **53,5 %** del archivo original. Esta reducción no implica por sí sola mala calidad: una parte importante queda afuera por definición del alcance. En cambio, las fechas invertidas sí constituyen un problema de consistencia. Los resultados sólo describen el subconjunto retenido y no deben generalizarse a pozos convencionales, verticales u otras cuencas.